# VIscaNEr — Transformer vs Fusion MLP on Manual-211

Evaluation only: no training. Predictions are generated for all 211 manual photos; closed-set retrieval metrics are reported for the 99 rows with reviewed catalog identities, including the original 72/27 Stage-2C split.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile

INPUT = Path('/kaggle/input')
PROJECT = Path('/kaggle/working/VIscaNEr_manual_eval')
OUTPUT = Path('/kaggle/working/manual_211_transformer_eval')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MPLBACKEND'] = 'Agg'

def input_root(slug):
    candidates = {path.resolve() for pattern in (slug, f'*/{slug}', f'*/*/{slug}') for path in INPUT.glob(pattern) if path.is_dir()}
    if len(candidates) != 1:
        raise RuntimeError(f'Attach exactly one Kaggle input {slug}; found: {sorted(candidates)}')
    return candidates.pop()

def required_dataset(slug, marker):
    root = input_root(slug)
    if not (root/marker).exists():
        raise FileNotFoundError(root/marker)
    return root

def unique_file(root, filename):
    matches = list(root.rglob(filename))
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one {filename} under {root}; found: {matches}')
    return matches[0]

print('SETUP | resolving Manual-211, code and completed checkpoints...', flush=True)
code = required_dataset('viscaner-fusion-stage2-code', 'code_sha256.json')
manual = required_dataset('viscaner-manual-211', 'build_summary.json')
labels = required_dataset('viscaner-dinov3-data', 'crops_metadata.csv')
bottles = required_dataset('viscaner-bottle-classifier-data', 'audit_metadata.csv')
label_checkpoint = unique_file(input_root('viscaner-dinov3-b-labels-hard-fine-tune'), 'best.pt')
bottle_checkpoint = unique_file(input_root('viscaner-dinov3-b-bottles-hard-fine-tune'), 'best.pt')
adapted_checkpoint = unique_file(input_root('viscaner-stage-2c-checkpoint-export'), 'manual_stage2c_best.pt')
transformer_checkpoint = unique_file(input_root('viscaner-fusion-transformer-ranker'), 'transformer_best.pt')

shutil.rmtree(PROJECT, ignore_errors=True)
shutil.copytree(code, PROJECT, ignore=shutil.ignore_patterns('dataset-metadata.json'))
for archive in PROJECT.glob('*.zip'):
    destination = PROJECT / archive.stem
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(destination)
checksums = json.loads((PROJECT/'code_sha256.json').read_text())
for name, expected in checksums.items():
    actual = hashlib.sha256((PROJECT/name).read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f'Code checksum mismatch: {name}')
sys.path.insert(0, str(PROJECT))
from fusion_stage2.kaggle_io import materialize_dataset_dirs
cache = Path('/kaggle/working/dataset_cache')
manual = materialize_dataset_dirs(manual, ('originals', 'label_crops', 'bottle_crops'), cache)
labels = materialize_dataset_dirs(labels, ('refs',), cache)
bottles = materialize_dataset_dirs(bottles, ('refs',), cache)
required = (manual/'manifest.csv', labels/'refs', bottles/'refs', label_checkpoint, bottle_checkpoint, adapted_checkpoint, transformer_checkpoint)
for path in required:
    if not path.exists():
        raise FileNotFoundError(path)
manual_summary = json.loads((manual/'build_summary.json').read_text())
if int(manual_summary.get('rows', 0)) != 211 or int(manual_summary.get('trainable_catalog_rows', 0)) != 99:
    raise RuntimeError(f'Unexpected Manual-211 summary: {manual_summary}')
print(f'SETUP | verified {len(checksums)} code files | all=211 | scored catalog=99', flush=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=5.6,<6', 'safetensors>=0.5,<1', 'rapidfuzz>=3,<4', 'Unidecode>=1.3,<2'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a Kaggle GPU accelerator before running this notebook')
print('SETUP | GPU:', torch.cuda.get_device_name(0), flush=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, '-u', '-m', 'fusion_stage2.evaluate_transformer_manual_211',
    '--manual-root', str(manual),
    '--label-refs-root', str(labels/'refs'), '--bottle-refs-root', str(bottles/'refs'),
    '--label-base-checkpoint', str(label_checkpoint), '--bottle-base-checkpoint', str(bottle_checkpoint),
    '--adapted-checkpoint', str(adapted_checkpoint), '--transformer-checkpoint', str(transformer_checkpoint),
    '--output-dir', str(OUTPUT), '--device', 'cuda', '--batch-size', '48', '--top-k', '10', '--report-top-k', '10',
]
print('EVALUATION | all 211 predictions; metrics on 99 catalog rows and held-out manual_val=27', flush=True)
subprocess.run(command, cwd=PROJECT, check=True)
summary = json.loads((OUTPUT/'manual_211_metrics.json').read_text())
print('FINAL MANUAL-211 TRANSFORMER EVAL |', json.dumps(summary, ensure_ascii=False), flush=True)
from IPython.display import Image as DisplayImage, display
display(DisplayImage(filename=str(OUTPUT/'manual_211_model_comparison.png')))
